# 成对实验

### 设置

In [ ]:
# 您可以在代码中直接设置
import os
os.environ["OPENAI_API_KEY"] = ""
os.environ["LANGSMITH_API_KEY"] = ""
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langsmith-academy"

In [ ]:
# 或者您可以使用 .env 文件
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

### 任务

让我们设置一个新任务！这里，我们有一个名叫鲍勃的销售员。鲍勃有很多交易，所以他想根据一些会议记录总结这些交易中发生的事情。

鲍勃正在迭代几个不同的提示词，这将为他的交易提供简洁明了的记录。

鲍勃整理了他的交易记录数据集，让我们加载它。如果您好奇的话，也可以查看这个数据集！请注意，这不是一个黄金数据集，这里没有参考输出。

In [ ]:
from langsmith import Client

client = Client()
dataset = client.clone_public_dataset(
  "https://smith.langchain.com/public/9078d2f1-7bef-4ba7-b795-210a17682ef9/d"
)

### 实验

现在，让我们使用两个不同的提示词在这个数据集上运行一些实验。让我们添加一个评估器来尝试评分我们的摘要有多好！

In [ ]:
from pydantic import BaseModel, Field
from openai import OpenAI

openai_client = OpenAI()

SUMMARIZATION_SYSTEM_PROMPT = """您是一名评判员，旨在评分摘要对记录内容的总结效果"""

SUMMARIZATION_HUMAN_PROMPT = """
[会议记录] {transcript}
[摘要开始] {summary} [摘要结束]"""

class SummarizationScore(BaseModel):
    score: int = Field(description="""1-5分评分，评估为提供的记录摘要的质量，1分为差摘要，5分为优秀摘要""")
    
def summary_score_evaluator(inputs: dict, outputs: dict) -> list:
    completion = openai_client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=[
            {   
                "role": "system",
                "content": SUMMARIZATION_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": SUMMARIZATION_HUMAN_PROMPT.format(
                    transcript=inputs["transcript"],
                    summary=outputs.get("output", "N/A"),
                )}
        ],
        response_format=SummarizationScore,
    )

    summary_score = completion.choices[0].message.parsed.score
    return {"key": "summary_score", "score": summary_score}

首先，我们将使用良好版本的提示词运行我们的实验！

In [ ]:
# 提示词一：良好提示词！
def good_summarizer(inputs: dict):
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": f"用3句话简洁地总结这次会议。确保包括所有重要事件。会议：{inputs['transcript']}"
            }
        ],
    )
    return response.choices[0].message.content

client.evaluate(
    good_summarizer,
    data=dataset,
    evaluators=[summary_score_evaluator],
    experiment_prefix="良好摘要器"
)

现在，我们将使用较差版本的提示词运行实验，以突出差异。

In [ ]:
# 提示词二：较差提示词！
def bad_summarizer(inputs: dict):
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": f"用一句话总结这个。{inputs['transcript']}"
            }
        ],
    )
    return response.choices[0].message.content

client.evaluate(
    bad_summarizer,
    data=dataset,
    evaluators=[summary_score_evaluator],
    experiment_prefix="较差摘要器"
)

### 成对实验

让我们定义一个将比较我们两个实验的函数。这些是成对评估器函数可以访问的字段：
- `inputs: dict`：对应数据集中单个示例的输入字典。
- `outputs: list[dict]`：每个实验在给定输入上产生的字典输出列表。
- `reference_outputs: dict`：与示例关联的参考输出字典（如果可用）。
- `runs: list[Run]`：实验在给定示例上生成的完整运行对象列表。如果您需要访问中间步骤或关于每次运行的元数据，请使用此项。
- `example: Example`：完整的数据集示例，包括示例输入、输出（如果可用）和元数据（如果可用）。

首先，让我们给我们的 LLM 评判员一些指示。在我们的情况下，我们将直接使用 LLM 评判员来评分哪个摘要器最有帮助。

在没有基准真相参考的情况下评分我们的摘要器可能很困难，但在这里，正面比较不同的提示词将让我们了解哪个更好！

In [ ]:
JUDGE_SYSTEM_PROMPT = """
请充当公正的评判员，评估两个AI摘要器对下面会议记录提供的摘要质量。
您的评估应考虑摘要的有用性、相关性、准确性、深度、创造性和详细程度等因素。
通过比较两个摘要开始您的评估，并提供简短解释。
避免任何立场偏见，确保呈现回答的顺序不影响您的决定。
不要偏爱某些助手的名称。
尽可能客观。"""

JUDGE_HUMAN_PROMPT = """
[会议记录] {transcript}

[助手A摘要开始] {answer_a} [助手A摘要结束]

[助手B摘要开始] {answer_b} [助手B摘要结束]"""

我们的函数将接收一个 `inputs` 字典和一个我们想要比较的不同实验的 `outputs` 字典列表。

In [ ]:
from pydantic import BaseModel, Field

class Preference(BaseModel):
    preference: int = Field(description="""如果助手A的答案基于上述因素更好，则为1。
如果助手B的答案基于上述因素更好，则为2。
如果平局则输出0。""")
    
def ranked_preference(inputs: dict, outputs: list[dict]) -> list:
    completion = openai_client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=[
            {   
                "role": "system",
                "content": JUDGE_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": JUDGE_HUMAN_PROMPT.format(
                    transcript=inputs["transcript"],
                    answer_a=outputs[0].get("output", "N/A"),
                    answer_b=outputs[1].get("output", "N/A")
                )}
        ],
        response_format=Preference,
    )

    preference_score = completion.choices[0].message.parsed.preference

    if preference_score == 1:
        scores = [1, 0]
    elif preference_score == 2:
        scores = [0, 1]
    else:
        scores = [0, 0]
    return scores

现在让我们使用 `evaluate()` 运行我们的成对实验

In [ ]:
from langsmith import evaluate

evaluate(
    ("Good Summarizer-bafea4ec", "Bad Summarizer-06ff299d"),  # TODO: 替换为您的实验名称/ID
    evaluators=[ranked_preference]
)